<a href="https://colab.research.google.com/github/shubham055555/Currency_Conversion_Tool_Using_ToolCalling/blob/main/Currency_Conversion_Tool_Using_ToolCalling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.environ["OPENROUTER_API_KEY"] = "sk-"

In [ ]:
!pip install -q langchain-openai langchain-core requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 3.2 MB/s eta 0:00:00


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool, InjectedToolArg
from langchain_core.messages import HumanMessage
from typing import Annotated
import requests
import json
import os

llm = ChatOpenAI(
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base="https://openrouter.ai/api/v1",
    model="nvidia/nemotron-3-super-120b-a12b:free",
)

In [ ]:
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'
  response = requests.get(url)
  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """
  return base_currency_value * conversion_rate

In [ ]:

print(get_conversion_factor.invoke({'base_currency': 'USD', 'target_currency': 'INR'}))
print(convert.invoke({'base_currency_value': 10, 'conversion_rate': 93.85}))

{'result': 'success', 'documentation': 'https://www.exchangerate-api.com/docs', 'terms_of_use': 'https://www.exchangerate-api.com/terms', 'time_last_update_unix': 1774137601, 'time_last_update_utc': 'Sun, 22 Mar 2026 00:00:01 +0000', 'time_next_update_unix': 1774224001, 'time_next_update_utc': 'Mon, 23 Mar 2026 00:00:01 +0000', 'base_code': 'USD', 'target_code': 'INR', 'conversion_rate': 93.6304}
938.5


In [ ]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [ ]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]






In [ ]:
ai_message = llm_with_tools.invoke(messages)

In [ ]:
messages.append(ai_message)

In [ ]:
print("Tool calls:", ai_message.tool_calls)

Tool calls: [{'name': 'convert', 'args': {'base_currency_value': 10}, 'id': 'call_722180949e9b4e52a998a64a', 'type': 'tool_call'}]


In [ ]:
for tool_call in ai_message.tool_calls:
 #tool1
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    messages.append(tool_message1)

  #tool2
  if tool_call['name'] == 'convert':
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)

In [ ]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 512, 'prompt_tokens': 818, 'total_tokens': 1330, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 562, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-super-120b-a12b-20230311:free', 'system_fingerprint': None, 'id': 'gen-1774204293-njhBwtCb2kWQWs6yXaFX', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d1

In [ ]:
llm_with_tools.invoke(messages).content

'The conversion factor from INR to USD is **0.01068 USD per INR**.  \n\nUsing that factor, **10 INR** converts to **0.1068 USD**.'

In [ ]:
final_response = llm_with_tools.invoke(messages).content
print(final_response)

The conversion factorfrom Indian Rupee (INR) to US Dollar (USD) is approximately **1 INR = 0.01068 USD**.

Based on this rate, converting **10 INR** to USD gives:  
**10 INR × 0.01068 USD/INR = 0.1068 USD**.

Therefore, **10 INR is equivalent to about 0.1068 USD**.

*Note: Exchange rates fluctuate constantly; this conversion uses the rate obtained at the time of the query.*
